# End-to-End Pipeline: Candidate Generation -> Attention Reranker -> Final K

This notebook runs the whole attention-reranker pipeline by itself. It does not require any
other notebook to be executed first, and it does not read or write intermediate candidate
files.

1. **Candidate generator**: trains the niche two-tower model from
   `CG_niche_content_two_tower_sequential_combined.ipynb` inside this notebook (or loads its
   checkpoint) and uses it to build 450 candidates per request: 200 rolling prior-hour
   popularity + 250 niche.
2. **Reranker**: trains the `RR_4` `+metadata` attention reranker inside this notebook (or
   loads its checkpoint) on the generator's training candidates.
3. **Evaluation**: generates the target-split candidates in memory, reranks them, and
   evaluates the final Top-K.

Saved artifacts are only the models, the reranker preprocessor, the causal-feature replay
cache, the final Top-K list and the metric tables. Candidate lists and scores stay in memory.

**Labels are incomplete.** Only articles the logging policy showed have a click label. A
generated article that was never shown has *unknown* relevance, so it is **not** a confirmed
negative. In evaluation it can never be credited as a click, and no denominator treats it as a
negative. It does, however, still occupy a Top-K slot and can push a known-click article down
or out. Every known click stays in the denominator, so a click the generator or reranker left
out of the Top-K is a miss.

**Training convention differs from evaluation.** The two listwise models use generated
candidates with no label as zero-labelled negatives during training. That is a known
false-negative risk. Evaluation still treats them as unknown, as described above.

## 0. Configure

**This is the only cell you need to edit.** The defaults run the full experiment on the whole
dataset. After editing, run the whole notebook with **Runtime > Run all**.

In [ ]:
# ---- Where to run ----------------------------------------------------------------------------
# "colab" = Google Colab with Google Drive, "local" = this computer, "auto" = detect automatically.
RUN_ENV = "auto"

# ---- Paths -------------------------------------------------------------------------------------
# Project folder: the shared folder holding the code (this repository).
# On Colab, add a shortcut to the shared folder in My Drive first; it may also sit one folder down.
COLAB_PROJECT_PATH = "/content/drive/MyDrive/BT4222-Group-6-"
LOCAL_PROJECT_PATH = None  # None = the repository this notebook is opened from

# Data path: the prepared data, set separately for Colab and local runs. It must contain
# 1_Data_Preparation/processed/, 3_Representations/artifacts/ and
# 4_Feature_Engineering/artifacts/prequential-all/.
# Metrics are written to <data path>/results/.
COLAB_DATA_PATH = None  # used on Colab; None = the project folder
LOCAL_DATA_PATH = (
    "/Users/numkoos/Downloads/v3-mindsmall"  # used locally; None = the project folder
)

# Model path: where the trained models are saved, and loaded from on later runs.
MODEL_PATH = None  # None = <data path>/5_Models/5_03_Pipeline_Evaluation/artifacts/attention_reranker_niche

# ---- Experiment --------------------------------------------------------------------------------
# Split to evaluate: "test" for the final results, or "validation".
TARGET_SPLIT = "test"

# Target impressions to evaluate. None = every impression in the split (default).
# A number, e.g. 5_000, evaluates a random sample instead for a quicker run.
EVAL_IMPRESSIONS = None

# Training impressions for the reranker. None = the whole training set (default).
# A number, e.g. 5_000, trains on a random sample instead for a quicker run.
TRAIN_IMPRESSIONS = None

# True: load a model saved by an earlier run when its settings match this cell (faster).
# False: always retrain from scratch.
REUSE_SAVED_MODELS = True

## 1.1 Load Packages

In [19]:
import copy
import json
import os
import sys
import time
from datetime import timedelta
from pathlib import Path
from types import SimpleNamespace

import joblib
import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import torch.nn.functional as F
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn

## 1.2 Set Up Paths

The run mode, project folder, data path and model path all come from the **Configure** cell
(section 0). On Colab the notebook mounts Google Drive and finds the shared project folder at
`COLAB_PROJECT_PATH`, or one folder down in My Drive. The data folders are checked up front, so a
wrong `DATA_PATH` fails here with the missing folder named.

In [20]:
# Run mode and paths come from the Configure cell (section 0)
ENVIRONMENT = RUN_ENV.lower()
if ENVIRONMENT == "auto":
    ENVIRONMENT = "colab" if "google.colab" in sys.modules else "local"
if ENVIRONMENT not in {"colab", "local"}:
    raise ValueError('RUN_ENV must be "auto", "colab" or "local"')

if ENVIRONMENT == "colab":
    from google.colab import drive

    drive.mount("/content/drive")
    requested = Path(COLAB_PROJECT_PATH)
    CODE_ROOT = next(
        (
            p
            for p in [
                requested,
                *Path("/content/drive/MyDrive").glob(f"*/{requested.name}"),
            ]
            if (p / "run_all.py").is_file()
        ),
        None,
    )
    if CODE_ROOT is None:
        raise FileNotFoundError(
            f"Could not find the project folder at {requested}. In Google Drive, right-click the shared "
            "folder > Organise > Add shortcut > My Drive (or fix COLAB_PROJECT_PATH), then rerun."
        )
elif LOCAL_PROJECT_PATH:
    CODE_ROOT = Path(LOCAL_PROJECT_PATH).expanduser().resolve()
else:
    CODE_ROOT = next(
        p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_all.py").is_file()
    )
ROOT = CODE_ROOT
DATA_PATH = COLAB_DATA_PATH if ENVIRONMENT == "colab" else LOCAL_DATA_PATH
DATA_ROOT = (
    Path(DATA_PATH or os.environ.get("NEWSREC_DATA_ROOT") or CODE_ROOT)
    .expanduser()
    .resolve()
)

REQUIRED_DATA = [
    "1_Data_Preparation/processed",
    "3_Representations/artifacts",
    "4_Feature_Engineering/artifacts/prequential-all",
]
missing_data = [
    str(DATA_ROOT / sub) for sub in REQUIRED_DATA if not (DATA_ROOT / sub).exists()
]
if missing_data:
    raise FileNotFoundError(
        f"{ENVIRONMENT.upper()}_DATA_PATH is missing these folders:\n"
        + "\n".join(missing_data)
    )
REPLAY_ROOT = DATA_ROOT

PROCESSED = REPLAY_ROOT / "1_Data_Preparation" / "processed"
REPRESENTATIONS = REPLAY_ROOT / "3_Representations" / "artifacts"
RUN = REPLAY_ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"
OUTPUT_DIR = REPLAY_ROOT / "5_Models" / "5_03_Pipeline_Evaluation" / "artifacts"
MODEL_DIR = (
    Path(MODEL_PATH).expanduser().resolve()
    if MODEL_PATH
    else OUTPUT_DIR / "attention_reranker_niche"
)
CG_MODEL_DIR = MODEL_DIR / "candidate_generator"
RR4_MODEL_DIR = MODEL_DIR / "reranker"
RESULTS = REPLAY_ROOT / "results"
REPLAY_CACHE_DIR = OUTPUT_DIR / "replayed_features"

sys.path.insert(0, str(ROOT / "5_Models" / "5_01_Candidate_Generation"))
sys.path.insert(0, str(ROOT / "5_Models" / "5_02_Reranking"))
sys.path.insert(0, str(ROOT / "src"))

import cg_evaluation
from cg_evaluation import CandidateEvaluator
from rr_evaluation import evaluate_retrieved_candidates

from newsrec.common import DEFAULT_SETTINGS
from newsrec.pipeline import replay_batches, time_horizon
from newsrec.ranking import TemporalState, build_features, feature_schema
from newsrec.representations import Bundle
from newsrec.training import iter_training_epoch

for path in (OUTPUT_DIR, CG_MODEL_DIR, RR4_MODEL_DIR, RESULTS, REPLAY_CACHE_DIR):
    path.mkdir(parents=True, exist_ok=True)

print("Repository:", ROOT)
print("Data path:", DATA_ROOT)
print("Model path:", MODEL_DIR)

Repository: C:\Users\Yong Rui\BT4222-Group-6-
Replay root: C:\Users\Yong Rui\BT4222-Group-6-
Model folder: C:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_03_Pipeline_Evaluation\artifacts\attention_reranker_niche


## 1.3 Configuration

In [21]:
TRAIN_SPLIT = "train"
K_VALUES = (5, 10, 30)
K = max(K_VALUES)
SEED = 4222

# Candidate generator (CG) settings
CG_SEED = 42
CG_POPULAR_TOP_N = 200
CG_NICHE_K = 250
CG_TOTAL = CG_POPULAR_TOP_N + CG_NICHE_K
CG_MAX_HISTORY = 50
CG_DIM = 64
CG_TEMPERATURE = 0.05
CG_FRESH_BUCKETS = 12
CG_NDCG_K = 300
CG_RECALL_K = 300
CG_N_SHOWN = 4
CG_BATCH = 512
CG_LEARNING_RATE = 1e-3
CG_EPOCHS = 20
CG_PATIENCE = 3
CG_EARLY_STOP_SAMPLE = 8_000
CG_SCORE_BATCH = 256
CG_HOUR_NS = 3_600 * 10**9
CG_TRAIN_ON = "all"
CG_LOSS = "bpr"
CG_SELECT_BY = "ndcg"
CG_REUSE_CHECKPOINT = REUSE_SAVED_MODELS
CG_NDCG_NAME = f"long_tail_ndcg@{CG_NDCG_K}"
CG_RECALL_NAME = f"long_tail_recall@{CG_RECALL_K}"
CG_SELECT_METRIC = CG_NDCG_NAME if CG_SELECT_BY == "ndcg" else CG_RECALL_NAME
CG_CONFIG = {
    "seed": CG_SEED,
    "popular_top_n": CG_POPULAR_TOP_N,
    "niche_k": CG_NICHE_K,
    "max_history": CG_MAX_HISTORY,
    "dim": CG_DIM,
    "temperature": CG_TEMPERATURE,
    "fresh_buckets": CG_FRESH_BUCKETS,
    "ndcg_k": CG_NDCG_K,
    "recall_k": CG_RECALL_K,
    "n_shown": CG_N_SHOWN,
    "batch": CG_BATCH,
    "learning_rate": CG_LEARNING_RATE,
    "epochs": CG_EPOCHS,
    "patience": CG_PATIENCE,
    "early_stop_sample": CG_EARLY_STOP_SAMPLE,
    "score_batch": CG_SCORE_BATCH,
    "train_on": CG_TRAIN_ON,
    "loss": CG_LOSS,
    "select_by": CG_SELECT_BY,
}
CG_CHECKPOINT_PATH = CG_MODEL_DIR / "model.pt"

# RR4 reranker settings
CONFIG = {
    "tabular": True,
    "text": True,
    "metadata": True,
    "title_encoder": False,
    "user_encoder": True,
    "position": False,
    "candidate_pooling": False,
    "short_term": False,
}
MODEL_DIM = 64
ATTENTION_HEADS = 2
ATTENTION_DROPOUT = 0.10
HIDDEN_DIMS = (256, 64)
DROPOUT = 0.20
NDCG_K = 10
LOSS_NAME = "softmax+lambda_ndcg"
HISTORY_CAP = 30
SHORT_TERM_CAP = 10
MAX_CANDIDATES = 100
HOLDOUT_FRACTION = 0.10
EPOCHS = 3
BATCH_IMPRESSIONS = 64
SCORING_BATCH_ROWS = 2048
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
GRAD_CLIP = 1.0
PATIENCE = 3
RESUME = True
RR4_REUSE_CHECKPOINT = REUSE_SAVED_MODELS
RR4_CHECKPOINT_PATH = RR4_MODEL_DIR / "model.pt"
PREPROCESSOR_PATH = RR4_MODEL_DIR / "preprocessor.joblib"

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
MODEL_NAME = "attention_reranker"
BASE_MODEL_NAME = "attention_reranker_niche_generated"
IMPRESSION_TAG = "all" if EVAL_IMPRESSIONS is None else f"{EVAL_IMPRESSIONS}imp"
CANDIDATE_CAP = CG_TOTAL

schema = json.loads((RUN / "feature_schema.json").read_text())
FEATURES = schema["model_columns"]
CATEGORICAL = [
    name for name in schema.get("categorical_columns", []) if name in FEATURES
]
NUMERIC = [name for name in FEATURES if name not in CATEGORICAL]
if feature_schema(DEFAULT_SETTINGS)["model_columns"] != FEATURES:
    raise ValueError(
        "The feature code and the saved feature run disagree on the schema"
    )


def rr4_checkpoint_meta():
    if not (
        RR4_REUSE_CHECKPOINT
        and RR4_CHECKPOINT_PATH.exists()
        and PREPROCESSOR_PATH.exists()
    ):
        return None
    meta = torch.load(RR4_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
    if (
        meta.get("features") != FEATURES
        or meta.get("config") != CONFIG
        or meta.get("train_impressions", "unknown") != TRAIN_IMPRESSIONS
    ):
        print(
            "RR4 checkpoint is incompatible with the current schema/config; retraining"
        )
        return None
    state_dict = meta.get("state_dict") or {}
    if not state_dict or not all(
        bool(torch.isfinite(value).all())
        for value in state_dict.values()
        if value.is_floating_point()
    ):
        print("RR4 checkpoint contains non-finite weights; retraining")
        return None
    return meta


RR4_CHECKPOINT_META = rr4_checkpoint_meta()
RR4_LOADED = RR4_CHECKPOINT_META is not None

print("Target split:", TARGET_SPLIT)
print("Final K:", K)
print("Evaluation impressions:", EVAL_IMPRESSIONS)
print("Device:", DEVICE)
print(
    f"Tabular features: {len(FEATURES)} ({len(NUMERIC)} numeric, {len(CATEGORICAL)} categorical)"
)
print("Candidate generator checkpoint:", CG_CHECKPOINT_PATH)
print("RR4 checkpoint:", RR4_CHECKPOINT_PATH)
print("RR4 checkpoint reusable:", RR4_LOADED)

Target split: test
Final K: 30
Evaluation impressions: None
Device: cuda
Tabular features: 50 (48 numeric, 2 categorical)
Candidate generator checkpoint: C:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_03_Pipeline_Evaluation\artifacts\attention_reranker_niche\candidate_generator\model.pt
RR4 checkpoint: C:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_03_Pipeline_Evaluation\artifacts\attention_reranker_niche\reranker\model.pt
RR4 checkpoint reusable: False


# 2. Candidate Generator

## 2.1 Article Features

In [22]:
index = pl.read_parquet(
    REPRESENTATIONS / "article_index.parquet",
    columns=["article_row", "news_code"],
).sort("article_row")

N_ITEMS = index.height + 1
ROWS = index["article_row"].to_numpy().astype(np.int64)
CODES = index["news_code"].to_numpy().astype(np.int64)
if not np.array_equal(ROWS, np.arange(1, N_ITEMS)):
    raise ValueError("Article rows must start from 1, with row 0 reserved for padding")

ROW_OF_CODE = np.zeros(int(CODES.max()) + 1, dtype=np.int64)
ROW_OF_CODE[CODES] = ROWS
CODE_OF_ROW = np.zeros(N_ITEMS, dtype=np.int32)
CODE_OF_ROW[ROWS] = CODES


def rows_for_codes(news_codes):
    news_codes = np.asarray(news_codes, dtype=np.int64)
    rows = np.zeros(news_codes.size, dtype=np.int64)
    valid = (news_codes >= 0) & (news_codes < ROW_OF_CODE.size)
    rows[valid] = ROW_OF_CODE[news_codes[valid]]
    return rows


TEXT = np.nan_to_num(np.load(REPRESENTATIONS / "text64.npy").astype(np.float32))
ENTITY = np.nan_to_num(
    np.load(REPRESENTATIONS / "article_entity100.npy").astype(np.float32)
)
if TEXT.shape[0] != N_ITEMS or ENTITY.shape[0] != N_ITEMS:
    raise ValueError("Article embeddings do not align with article_index")

availability = pl.read_parquet(
    REPRESENTATIONS / "article_availability.parquet",
    columns=["news_code", "available_from"],
)
AVAIL_NS = np.full(N_ITEMS, np.iinfo(np.int64).max, dtype=np.int64)
avail_rows = rows_for_codes(availability["news_code"].to_numpy())
avail_times = availability["available_from"].dt.epoch("ns").to_numpy()
valid = avail_rows > 0
AVAIL_NS[avail_rows[valid]] = avail_times[valid]

catalogue = pl.read_parquet(
    PROCESSED / "news_catalogue.parquet",
    columns=["news_code", "category", "subcategory"],
)
catalogue_rows = rows_for_codes(catalogue["news_code"].to_numpy())


def encode_labels(column):
    labels = catalogue[column].cast(pl.Utf8).fill_null("").to_list()
    unique_labels = sorted(set(labels) - {""})
    lookup = {label: i + 1 for i, label in enumerate(unique_labels)}
    encoded = np.array([lookup.get(label, 0) for label in labels], dtype=np.int64)
    codes = np.zeros(N_ITEMS, dtype=np.int64)
    valid = catalogue_rows > 0
    codes[catalogue_rows[valid]] = encoded[valid]
    return codes, len(lookup) + 1


CATEGORY, N_CATEGORY = encode_labels("category")
SUBCATEGORY, N_SUBCATEGORY = encode_labels("subcategory")
print(
    f"Articles: {N_ITEMS - 1:,} | Categories: {N_CATEGORY - 1} | "
    f"Subcategories: {N_SUBCATEGORY - 1}"
)

Articles: 51,282 | Categories: 17 | Subcategories: 264


## 2.2 Requests, Clicks and Popularity

In [23]:
first_ts = (
    pl.scan_parquet(PROCESSED / "candidates_train.parquet")
    .select(pl.col("timestamp").min())
    .collect()
    .item()
)
T0_NS = int(
    np.datetime64(first_ts, "ns")
    .astype("datetime64[D]")
    .astype("datetime64[ns]")
    .astype(np.int64)
)


def load_requests(split):
    impressions = pl.read_parquet(
        PROCESSED / f"impressions_{split}.parquet",
        columns=["impression_uid", "timestamp"],
    )
    files = sorted((RUN / f"request_profiles_{split}").rglob("*.parquet"))
    if not files:
        raise FileNotFoundError(f"No request_profiles_{split} under {RUN}")
    history_col = (
        "full_history_rows"
        if "full_history_rows" in pl.read_parquet_schema(files[0])
        else "history_rows"
    )
    requests = impressions.join(
        pl.read_parquet(files, columns=["impression_uid", history_col]),
        on="impression_uid",
        how="left",
    ).sort("timestamp", "impression_uid")
    history = requests[history_col].list.eval(pl.element().filter(pl.element() > 0))
    lengths = history.list.len().fill_null(0).to_numpy().astype(np.int64)
    flat = history.explode().drop_nulls().to_numpy().astype(np.int64)
    offsets = np.r_[0, np.cumsum(lengths)]
    take = np.minimum(lengths, CG_MAX_HISTORY)
    position = np.arange(CG_MAX_HISTORY)[None, :]
    start = (offsets[1:] - take)[:, None]
    tail = np.where(
        position < take[:, None],
        np.r_[flat, 0][np.minimum(start + position, flat.size)],
        0,
    )
    ts = requests["timestamp"].dt.epoch("ns").to_numpy().astype(np.int64)
    return SimpleNamespace(
        name=split,
        uid=requests["impression_uid"].to_numpy().astype(np.int64),
        ts=ts,
        hour=(ts - T0_NS) // CG_HOUR_NS,
        hist_tail=tail.astype(np.int64),
        hist_flat=flat,
        hist_off=offsets,
    )


def load_clicks(requests):
    lookup = pl.DataFrame(
        {"impression_uid": requests.uid, "req": np.arange(requests.uid.size)}
    )
    clicks = (
        pl.scan_parquet(PROCESSED / f"candidates_{requests.name}.parquet")
        .filter(pl.col("clicked") == 1)
        .select("impression_uid", "news_code")
        .unique()
        .collect()
        .join(lookup, on="impression_uid")
        .sort("req")
    )
    rows = rows_for_codes(clicks["news_code"].to_numpy())
    if (rows == 0).any():
        raise ValueError(
            f"{requests.name}: clicks on articles missing from article_index"
        )
    return clicks["req"].to_numpy().astype(np.int64), rows


TRAIN = load_requests("train")
VAL = load_requests("validation")
TEST = load_requests("test")
TRAIN_CLICK_REQ, TRAIN_CLICK_ROW = load_clicks(TRAIN)
VAL_CLICK_REQ, VAL_CLICK_ROW = load_clicks(VAL)

hourly = (
    pl.concat(
        [
            pl.scan_parquet(PROCESSED / f"candidates_{split}.parquet").select(
                "news_code",
                "clicked",
                ((pl.col("timestamp").dt.epoch("ns") - T0_NS) // CG_HOUR_NS).alias(
                    "hour"
                ),
            )
            for split in ("train", "validation", "test")
        ]
    )
    .group_by("news_code", "hour")
    .agg(shows=pl.len(), clicks=pl.col("clicked").cast(pl.Int64).sum())
    .collect()
)
N_HOURS = (
    int(max(hourly["hour"].max(), TRAIN.hour.max(), VAL.hour.max(), TEST.hour.max()))
    + 1
)
EXPOSURES = np.zeros((N_HOURS, N_ITEMS), dtype=np.int32)
CLICKS = np.zeros((N_HOURS, N_ITEMS), dtype=np.int32)
hourly_rows = rows_for_codes(hourly["news_code"].to_numpy())
hours = hourly["hour"].to_numpy()
EXPOSURES[hours, hourly_rows] = hourly["shows"].to_numpy()
CLICKS[hours, hourly_rows] = hourly["clicks"].to_numpy()
del hourly


def popular_order(hour):
    eligible = np.flatnonzero(AVAIL_NS <= T0_NS + (hour + 1) * CG_HOUR_NS)
    if hour == 0:
        clicks = shows = np.zeros(N_ITEMS, dtype=np.int32)
    else:
        clicks, shows = CLICKS[hour - 1], EXPOSURES[hour - 1]
    return eligible[
        np.lexsort((CODE_OF_ROW[eligible], -shows[eligible], -clicks[eligible]))
    ]


POPULAR_MASK = np.zeros((N_HOURS, N_ITEMS), dtype=bool)
for hour in range(N_HOURS):
    POPULAR_MASK[hour, popular_order(hour)[:CG_POPULAR_TOP_N]] = True

VAL_POPULAR = np.zeros((VAL.uid.size, CG_POPULAR_TOP_N), dtype=np.int64)
for hour in np.unique(VAL.hour):
    idx = np.flatnonzero(VAL.hour == hour)
    ranked = popular_order(hour)[:CG_POPULAR_TOP_N]
    if ranked.size < CG_POPULAR_TOP_N:
        raise ValueError(
            "Fewer than the requested popular articles for a validation hour"
        )
    VAL_POPULAR[idx] = ranked

on_shelf = (VAL_POPULAR[VAL_CLICK_REQ] == VAL_CLICK_ROW[:, None]).any(axis=1)
NICHE_REQ = VAL_CLICK_REQ[~on_shelf]
NICHE_ROW = VAL_CLICK_ROW[~on_shelf]
NICHE_IMPRESSIONS = np.unique(NICHE_REQ)
print(
    f"Validation clicks: {VAL_CLICK_REQ.size:,} | niche clicks: {NICHE_REQ.size:,} | "
    f"niche impressions: {NICHE_IMPRESSIONS.size:,}"
)

C:\Users\Yong Rui\AppData\Local\Temp\ipykernel_14980\3602891738.py:36: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  flat = history.explode().drop_nulls().to_numpy().astype(np.int64)


Validation clicks: 47,961 | niche clicks: 9,778 | niche impressions: 7,828


## 2.3 Niche Two-Tower Model and Retrieval

In [24]:
AVAIL_T = torch.from_numpy(AVAIL_NS).to(DEVICE)


def top_niche(score_fn, requests, popular_rows, depth, subset=None, allow_short=False):
    subset = np.arange(requests.uid.size) if subset is None else np.asarray(subset)
    out = np.zeros((subset.size, depth), dtype=np.int64)
    short = np.zeros(subset.size, dtype=bool)
    for start in range(0, subset.size, CG_SCORE_BATCH):
        idx = subset[start : start + CG_SCORE_BATCH]
        batch = np.arange(idx.size)
        with torch.no_grad():
            scores = score_fn(idx).float().clone()
        scores[
            AVAIL_T[None, :] > torch.from_numpy(requests.ts[idx]).to(DEVICE)[:, None]
        ] = -torch.inf
        scores[
            torch.from_numpy(np.repeat(batch, popular_rows.shape[1])).to(DEVICE),
            torch.from_numpy(popular_rows[idx].ravel()).to(DEVICE),
        ] = -torch.inf
        lengths = requests.hist_off[idx + 1] - requests.hist_off[idx]
        read = np.concatenate(
            [
                requests.hist_flat[requests.hist_off[i] : requests.hist_off[i + 1]]
                for i in idx
            ]
        )
        scores[
            torch.from_numpy(np.repeat(batch, lengths)).to(DEVICE),
            torch.from_numpy(read).to(DEVICE),
        ] = -torch.inf
        values, top = torch.topk(scores, depth, dim=1)
        if torch.isinf(values).any():
            if not allow_short:
                raise ValueError(
                    "Fewer niche articles available than the requested depth"
                )
            short[start : start + idx.size] = (
                torch.isinf(values).any(dim=1).cpu().numpy()
            )
        out[start : start + idx.size] = top.cpu().numpy()
    return (out, short) if allow_short else out


def niche_metrics(top_rows, subset=None):
    subset = np.arange(VAL.uid.size) if subset is None else np.asarray(subset)
    n, _depth = top_rows.shape
    position = np.full(VAL.uid.size, -1, dtype=np.int64)
    position[subset] = np.arange(n)
    keep = position[NICHE_REQ] >= 0
    click_req = position[NICHE_REQ[keep]]
    click_row = NICHE_ROW[keep]
    match = top_rows[click_req] == click_row[:, None]
    found = match.any(axis=1)
    rank = np.where(found, match.argmax(axis=1) + 1, 0)
    hit = found & (rank <= CG_NDCG_K)
    dcg = np.bincount(click_req[hit], weights=1 / np.log2(rank[hit] + 1), minlength=n)
    relevant = np.bincount(click_req, minlength=n)
    ideal = np.r_[0, np.cumsum(1 / np.log2(np.arange(2, CG_NDCG_K + 2)))][
        np.minimum(relevant, CG_NDCG_K)
    ]
    has_click = relevant > 0
    return {
        CG_NDCG_NAME: float((dcg[has_click] / ideal[has_click]).mean()),
        CG_RECALL_NAME: float((found & (rank <= CG_RECALL_K)).mean()),
    }


TEXT_T = torch.from_numpy(TEXT).to(DEVICE)
ENTITY_T = torch.from_numpy(ENTITY).to(DEVICE)
CATEGORY_T = torch.from_numpy(CATEGORY).to(DEVICE)
SUBCATEGORY_T = torch.from_numpy(SUBCATEGORY).to(DEVICE)
ALL_ROWS = torch.arange(N_ITEMS, device=DEVICE)


class ArticleTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.category = nn.Embedding(N_CATEGORY, 16, padding_idx=0)
        self.subcategory = nn.Embedding(N_SUBCATEGORY, 16, padding_idx=0)
        self.network = nn.Sequential(
            nn.Linear(TEXT.shape[1] + ENTITY.shape[1] + 32, 128),
            nn.ReLU(),
            nn.Linear(128, CG_DIM),
        )

    def forward(self, rows):
        x = torch.cat(
            [
                TEXT_T[rows],
                ENTITY_T[rows],
                self.category(CATEGORY_T[rows]),
                self.subcategory(SUBCATEGORY_T[rows]),
            ],
            dim=-1,
        )
        return F.normalize(self.network(x), dim=-1)


class UserTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(CG_DIM, CG_DIM), nn.Tanh(), nn.Linear(CG_DIM, 1)
        )
        self.project = nn.Linear(CG_DIM, CG_DIM)
        self.new_user = nn.Parameter(torch.randn(CG_DIM) * 0.01)

    def forward(self, history_vectors, mask):
        logits = self.attention(history_vectors).squeeze(-1).masked_fill(~mask, -1e9)
        weights = torch.softmax(logits, dim=1) * mask
        pooled = (weights.unsqueeze(-1) * history_vectors).sum(1)
        pooled = torch.where(
            mask.any(1, keepdim=True), pooled, self.new_user.expand_as(pooled)
        )
        return F.normalize(pooled + self.project(pooled), dim=-1)


class NicheTwoTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.article = ArticleTower()
        self.user = UserTower()
        self.freshness = nn.Embedding(CG_FRESH_BUCKETS, 1)
        nn.init.zeros_(self.freshness.weight)

    def article_vectors(self):
        return self.article(ALL_ROWS)

    def user_vectors(self, hist, article_vectors):
        hist = torch.as_tensor(hist, device=article_vectors.device)
        return self.user(article_vectors[hist], hist > 0)

    def freshness_bonus(self, ts, rows):
        ts = torch.as_tensor(ts, device=AVAIL_T.device)
        rows = torch.as_tensor(rows, device=AVAIL_T.device)
        age_hours = (ts[:, None] - AVAIL_T[rows]).clamp(min=0).float() / CG_HOUR_NS
        bucket = torch.log2(1 + age_hours).floor().long().clamp(0, CG_FRESH_BUCKETS - 1)
        return self.freshness(bucket).squeeze(-1)

    def score(self, users, article_vectors, ts, rows):
        rows = torch.as_tensor(rows, device=article_vectors.device)
        vectors = article_vectors[rows]
        similarity = (
            users @ vectors.T
            if vectors.dim() == 2
            else (users.unsqueeze(1) * vectors).sum(-1)
        )
        return similarity / CG_TEMPERATURE + self.freshness_bonus(ts, rows)


def model_scorer(net, article_vectors, requests):
    def score_fn(idx):
        users = net.user_vectors(requests.hist_tail[idx], article_vectors)
        return net.score(users, article_vectors, requests.ts[idx], ALL_ROWS)

    return score_fn

## 2.4 Training Examples and Negative Sampling

In [25]:
RNG = np.random.default_rng(CG_SEED)

in_history = np.isin(
    TRAIN_CLICK_REQ * N_ITEMS + TRAIN_CLICK_ROW,
    np.repeat(np.arange(TRAIN.uid.size), np.diff(TRAIN.hist_off)) * N_ITEMS
    + TRAIN.hist_flat,
)
on_shelf_train = POPULAR_MASK[TRAIN.hour[TRAIN_CLICK_REQ], TRAIN_CLICK_ROW]
keep = ~in_history & (~on_shelf_train if CG_TRAIN_ON == "niche_only" else True)
POS_REQ, POS_ROW = TRAIN_CLICK_REQ[keep], TRAIN_CLICK_ROW[keep]
EXAMPLE_UID = TRAIN.uid[POS_REQ]


def draw_shown(epoch):
    shown = pl.concat(
        [
            pl.from_pandas(
                frame.loc[frame["clicked"] == 0, ["impression_uid", "news_code"]]
            )
            for frame, _ in iter_training_epoch(RUN, epoch, columns=[])
        ]
    )
    pairs = pl.DataFrame(
        {"impression_uid": EXAMPLE_UID, "example": np.arange(EXAMPLE_UID.size)}
    ).join(
        shown.with_columns(pl.col("impression_uid").cast(pl.Int64)), on="impression_uid"
    )
    pairs = (
        pairs.with_columns(
            pl.Series(
                "order", np.random.default_rng([CG_SEED, epoch]).random(pairs.height)
            )
        )
        .sort("example", "order")
        .with_columns(pl.int_range(pl.len()).over("example").alias("slot"))
        .filter(pl.col("slot") < CG_N_SHOWN)
    )
    negatives = np.zeros((EXAMPLE_UID.size, CG_N_SHOWN), dtype=np.int64)
    negatives[pairs["example"].to_numpy(), pairs["slot"].to_numpy()] = rows_for_codes(
        pairs["news_code"].to_numpy()
    )
    return negatives


def sample_negatives(examples, shown_neg):
    req = POS_REQ[examples]
    positive = POS_ROW[examples]
    negatives = shown_neg[examples]
    ignore = negatives == 0
    ignore |= negatives == positive[:, None]
    ignore |= (negatives[:, :, None] == TRAIN.hist_tail[req][:, None, :]).any(-1)
    return negatives, ignore


def training_loss(logits, ignore):
    valid = ~torch.from_numpy(ignore).to(logits.device)
    if CG_LOSS == "softmax":
        masked = torch.cat(
            [logits[:, :1], logits[:, 1:].masked_fill(~valid, -torch.inf)], dim=1
        )
        return F.cross_entropy(
            masked, torch.zeros(logits.shape[0], dtype=torch.long, device=logits.device)
        )
    pairs = F.softplus(logits[:, 1:] - logits[:, :1])[valid]
    return pairs.mean() if pairs.numel() else logits.sum() * 0.0

## 2.5 Train or Load the Candidate Generator

In [26]:
def train_candidate_generator():
    rng = np.random.default_rng(CG_SEED)
    model = NicheTwoTower().to(DEVICE)
    optimiser = torch.optim.Adam(model.parameters(), lr=CG_LEARNING_RATE)
    early_stop_requests = np.sort(
        rng.choice(
            NICHE_IMPRESSIONS,
            size=min(CG_EARLY_STOP_SAMPLE, NICHE_IMPRESSIONS.size),
            replace=False,
        )
    )
    check_depth = max(CG_NDCG_K, CG_RECALL_K)
    best_score = -1.0
    best_state = None
    best_epoch = 0
    no_improvement = 0
    shown_neg = draw_shown(0)

    for epoch in range(1, CG_EPOCHS + 1):
        started = time.time()
        if epoch > 1:
            shown_neg = draw_shown(epoch - 1)
        model.train()
        losses = []
        order = rng.permutation(POS_REQ.size)
        for start in range(0, order.size, CG_BATCH):
            batch = order[start : start + CG_BATCH]
            req = POS_REQ[batch]
            positive = POS_ROW[batch]
            negatives, ignore = sample_negatives(batch, shown_neg)
            article_vectors = model.article_vectors()
            users = model.user_vectors(TRAIN.hist_tail[req], article_vectors)
            candidates = torch.from_numpy(
                np.concatenate([positive[:, None], negatives], axis=1)
            ).to(DEVICE)
            loss = training_loss(
                model.score(users, article_vectors, TRAIN.ts[req], candidates), ignore
            )
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
            losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            lists = top_niche(
                model_scorer(model, model.article_vectors(), VAL),
                VAL,
                VAL_POPULAR,
                check_depth,
                subset=early_stop_requests,
            )
        scores = niche_metrics(lists, subset=early_stop_requests)
        improved = scores[CG_SELECT_METRIC] > best_score
        if improved:
            best_score = scores[CG_SELECT_METRIC]
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            no_improvement = 0
        else:
            no_improvement += 1
        print(
            f"cg epoch {epoch:02d} | {CG_LOSS} loss {np.mean(losses):.4f} | "
            f"{CG_NDCG_NAME} {scores[CG_NDCG_NAME]:.4f} | "
            f"{CG_RECALL_NAME} {scores[CG_RECALL_NAME]:.4f} | "
            f"{time.time() - started:.0f}s"
            f"{' | BEST' if improved else f' | patience {no_improvement}/{CG_PATIENCE}'}"
        )
        if no_improvement >= CG_PATIENCE:
            print(f"cg early stopping after epoch {epoch}")
            break

    model.load_state_dict(best_state)
    model.eval()
    return model, best_epoch, best_score


CG_LOADED = False
CG_BEST_EPOCH = None
CG_BEST_SCORE = None
if CG_REUSE_CHECKPOINT and CG_CHECKPOINT_PATH.exists():
    cg_meta = torch.load(CG_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
    if (
        cg_meta.get("config") == CG_CONFIG
        and cg_meta.get("n_items") == N_ITEMS
        and cg_meta.get("n_category") == N_CATEGORY
        and cg_meta.get("n_subcategory") == N_SUBCATEGORY
    ):
        cg_model = NicheTwoTower().to(DEVICE)
        cg_model.load_state_dict(cg_meta["state_dict"])
        cg_model.eval()
        CG_LOADED = True
        CG_BEST_EPOCH = cg_meta.get("best_epoch")
        CG_BEST_SCORE = cg_meta.get("best_score")
        print("Loaded candidate generator:", CG_CHECKPOINT_PATH)
    else:
        print("Candidate generator checkpoint config/shape mismatch; retraining")

if not CG_LOADED:
    cg_model, CG_BEST_EPOCH, CG_BEST_SCORE = train_candidate_generator()
    torch.save(
        {
            "state_dict": {
                key: value.detach().cpu()
                for key, value in cg_model.state_dict().items()
            },
            "config": CG_CONFIG,
            "n_items": N_ITEMS,
            "n_category": N_CATEGORY,
            "n_subcategory": N_SUBCATEGORY,
            "best_epoch": CG_BEST_EPOCH,
            "best_score": CG_BEST_SCORE,
            "select_metric": CG_SELECT_METRIC,
        },
        CG_CHECKPOINT_PATH,
    )
    print("Saved candidate generator:", CG_CHECKPOINT_PATH)

Loaded candidate generator: C:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_03_Pipeline_Evaluation\artifacts\attention_reranker_niche\candidate_generator\model.pt


# 3. Generate Candidates in Memory

In [27]:
def subset_requests(requests, indices):
    tail = requests.hist_tail[indices]
    lengths = (tail > 0).sum(axis=1)
    flat = tail[tail > 0]
    offsets = np.r_[0, np.cumsum(lengths)]
    return SimpleNamespace(
        name=requests.name,
        uid=requests.uid[indices],
        ts=requests.ts[indices],
        hour=requests.hour[indices],
        hist_tail=tail,
        hist_flat=flat,
        hist_off=offsets,
    )


def popular_rows_for(requests, filter_history=True):
    out = np.zeros((requests.uid.size, CG_POPULAR_TOP_N), dtype=np.int64)
    short = np.zeros(requests.uid.size, dtype=bool)
    for hour in np.unique(requests.hour):
        idx = np.flatnonzero(requests.hour == hour)
        ranked = popular_order(hour)[: CG_POPULAR_TOP_N + 2_000]
        ok = AVAIL_NS[ranked][None, :] <= requests.ts[idx][:, None]
        if filter_history:
            lengths = requests.hist_off[idx + 1] - requests.hist_off[idx]
            if lengths.sum():
                read_rows = np.concatenate(
                    [
                        requests.hist_flat[
                            requests.hist_off[i] : requests.hist_off[i + 1]
                        ]
                        for i in idx
                    ]
                )
                read = np.isin(
                    np.arange(idx.size)[:, None] * N_ITEMS + ranked[None, :],
                    np.repeat(np.arange(idx.size), lengths) * N_ITEMS + read_rows,
                )
                ok &= ~read
        take = ok & (np.cumsum(ok, axis=1) <= CG_POPULAR_TOP_N)
        full = take.sum(axis=1) == CG_POPULAR_TOP_N
        out[idx[full]] = ranked[np.nonzero(take[full])[1]].reshape(-1, CG_POPULAR_TOP_N)
        short[idx[~full]] = True
    return out, short


def generate_candidates(requests, cg_model):
    popular, popular_short = popular_rows_for(requests)
    niche, niche_short = top_niche(
        model_scorer(cg_model, cg_model.article_vectors(), requests),
        requests,
        popular,
        CG_NICHE_K,
        allow_short=True,
    )
    keep = ~(popular_short | niche_short)
    if not keep.any():
        raise ValueError("No complete candidate lists were generated")
    rows = np.concatenate([popular[keep], niche[keep]], axis=1)
    ordered = np.sort(rows, axis=1)
    if (ordered[:, 1:] == ordered[:, :-1]).any():
        raise ValueError("Duplicate articles in a generated candidate list")
    uid = requests.uid[keep]
    total = rows.shape[1]
    return pl.DataFrame(
        {
            "impression_uid": np.repeat(uid, total).astype(np.int64),
            "cutoff": np.tile(np.arange(1, total + 1, dtype=np.int16), uid.size),
            "news_code": CODE_OF_ROW[rows].reshape(-1).astype(np.int64),
        }
    ).sort(["impression_uid", "cutoff"])


target_requests = VAL if TARGET_SPLIT == "validation" else TEST
if EVAL_IMPRESSIONS is None:
    target_indices = np.arange(target_requests.uid.size)
else:
    chosen = np.random.default_rng(SEED).choice(
        target_requests.uid,
        size=min(EVAL_IMPRESSIONS, target_requests.uid.size),
        replace=False,
    )
    target_indices = np.flatnonzero(np.isin(target_requests.uid, np.sort(chosen)))
target_gen_requests = subset_requests(target_requests, target_indices)
target_candidate_frame = generate_candidates(target_gen_requests, cg_model)
selected_uids = target_candidate_frame["impression_uid"].unique().sort()
n_selected = selected_uids.len()
print(
    f"Target candidates: {n_selected:,} impressions / "
    f"{target_candidate_frame.height:,} rows"
)

if RR4_LOADED:
    train_candidate_frame = None
    print("RR4 checkpoint reused; skipping training-candidate generation")
else:
    if TRAIN_IMPRESSIONS is None:
        train_indices = np.arange(TRAIN.uid.size)
    else:
        chosen_train = np.random.default_rng(SEED).choice(
            TRAIN.uid,
            size=min(TRAIN_IMPRESSIONS, TRAIN.uid.size),
            replace=False,
        )
        train_indices = np.flatnonzero(np.isin(TRAIN.uid, np.sort(chosen_train)))
    train_gen_requests = subset_requests(TRAIN, train_indices)
    train_candidate_frame = generate_candidates(train_gen_requests, cg_model)
    print(
        f"Training candidates: "
        f"{train_candidate_frame['impression_uid'].n_unique():,} impressions / "
        f"{train_candidate_frame.height:,} rows"
    )

Target candidates: 30,270 impressions / 13,621,500 rows
Training candidates: 5,000 impressions / 2,250,000 rows


# 4. Train the RR4 Reranker

## 4.1 Label and Cap the Generated Training Candidates

In [28]:
def label_generated_candidates(candidates, split):
    label_files = sorted((RUN / f"labels_{split}").glob("*.parquet"))
    if not label_files:
        raise FileNotFoundError(f"No labels_{split} partitions in {RUN}")
    uids = [int(uid) for uid in candidates["impression_uid"].unique()]
    labels = (
        pl.scan_parquet(label_files)
        .filter(pl.col("impression_uid").is_in(uids))
        .select("impression_uid", "news_code", "clicked")
        .with_columns(
            pl.col("impression_uid").cast(pl.Int64),
            pl.col("news_code").cast(pl.Int64),
            pl.col("clicked").cast(pl.Int8),
        )
        .collect()
    )
    labeled = candidates.to_pandas().merge(
        labels.to_pandas(),
        on=["impression_uid", "news_code"],
        how="left",
        validate="one_to_one",
    )
    if labeled["impression_uid"].nunique() != candidates["impression_uid"].n_unique():
        raise ValueError("Some generated training impressions have no labels")
    labeled["clicked"] = labeled["clicked"].fillna(0).astype(np.int8)
    return labeled


def cap_slates(frame, max_candidates):
    rng = np.random.default_rng(SEED)
    keep = []
    for rows in frame.groupby("impression_uid", sort=False).indices.values():
        if len(rows) <= max_candidates:
            keep.append(rows)
            continue
        clicked = rows[frame["clicked"].to_numpy()[rows] == 1]
        unclicked = rows[frame["clicked"].to_numpy()[rows] == 0]
        n_negatives = max(max_candidates - len(clicked), 1)
        keep.append(
            np.concatenate(
                [
                    clicked,
                    rng.choice(
                        unclicked, size=min(n_negatives, len(unclicked)), replace=False
                    ),
                ]
            )
        )
    return frame.iloc[np.sort(np.concatenate(keep))].reset_index(drop=True)


if not RR4_LOADED:
    train_labeled = label_generated_candidates(train_candidate_frame, TRAIN_SPLIT)
    slate_stats = train_labeled.groupby("impression_uid")["clicked"].agg(
        ["sum", "size"]
    )
    usable = slate_stats.index[
        (slate_stats["sum"] > 0) & (slate_stats["sum"] < slate_stats["size"])
    ]
    train_labeled = train_labeled[
        train_labeled["impression_uid"].isin(usable)
    ].reset_index(drop=True)
    if not len(train_labeled):
        raise ValueError(
            "No generated training impression has both a click and a non-click"
        )
    train_slates = cap_slates(train_labeled, MAX_CANDIDATES)
    print(
        f"Usable generated train slates: {train_slates['impression_uid'].nunique():,} "
        f"impressions / {len(train_slates):,} capped rows "
        f"(click rate {train_slates['clicked'].mean():.4f})"
    )
else:
    train_slates = None

Usable generated train slates: 4,456 impressions / 445,600 capped rows (click rate 0.0139)


## 4.2 RR4 Model and Causal Feature Replay

In [29]:
class ArticleInputs:
    def __init__(self, article_index, text, entity, title_tokens):
        self.news_to_row = dict(
            zip(
                article_index["news_code"].to_list(),
                article_index["article_row"].to_list(),
            )
        )
        self.news_to_row = {int(k): int(v) for k, v in self.news_to_row.items()}
        self.text = text
        self.entity = entity
        self.title_tokens = title_tokens


def load_article_inputs():
    article_index = pl.read_parquet(
        REPRESENTATIONS / "article_index.parquet",
        columns=["news_code", "article_row", "category_code", "subcategory_code"],
    )
    text = np.array(
        np.load(REPRESENTATIONS / "text64.npy", mmap_mode="r"), dtype=np.float32
    )
    entity = np.array(
        np.load(REPRESENTATIONS / "article_entity100.npy", mmap_mode="r"),
        dtype=np.float32,
    )
    title_tokens = np.load(REPRESENTATIONS / "title_token_ids.npy").astype(np.int64)
    inputs = ArticleInputs(article_index, text, entity, title_tokens)
    inputs.article_category = np.zeros(text.shape[0], dtype=np.int64)
    inputs.article_subcategory = np.zeros(text.shape[0], dtype=np.int64)
    inputs.article_category[article_index["article_row"].to_numpy()] = article_index[
        "category_code"
    ].to_numpy()
    inputs.article_subcategory[article_index["article_row"].to_numpy()] = article_index[
        "subcategory_code"
    ].to_numpy()
    return inputs


class AdditiveAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.proj = nn.Linear(dim, dim)
        self.query = nn.Parameter(torch.randn(dim) * 0.02)

    def forward(self, states, padding_mask):
        all_padded = padding_mask.all(dim=1)
        safe_mask = padding_mask.clone()
        safe_mask[all_padded, 0] = False
        scores = torch.tanh(self.proj(states)) @ self.query
        weights = torch.softmax(scores.masked_fill(safe_mask, -1e9), dim=1)
        weights = weights.masked_fill(padding_mask, 0.0)
        weights = weights / weights.sum(dim=1, keepdim=True).clamp(min=1e-8)
        return (weights.unsqueeze(-1) * states).sum(dim=1)


class NewsEncoder(nn.Module):
    def __init__(self, config, inputs):
        super().__init__()
        self.config = config
        self.register_buffer("text", torch.from_numpy(inputs.text), persistent=False)
        width = self.text.shape[1]

        if config["metadata"]:
            self.register_buffer(
                "entity", torch.from_numpy(inputs.entity), persistent=False
            )
            self.register_buffer(
                "category", torch.from_numpy(inputs.article_category), persistent=False
            )
            self.register_buffer(
                "subcategory",
                torch.from_numpy(inputs.article_subcategory),
                persistent=False,
            )
            self.category_embedding = nn.Embedding(
                int(inputs.article_category.max()) + 1, 16, padding_idx=0
            )
            self.subcategory_embedding = nn.Embedding(
                int(inputs.article_subcategory.max()) + 1, 16, padding_idx=0
            )
            self.entity_projection = nn.Linear(self.entity.shape[1], 32)
            width += 16 + 16 + 32

        if config["title_encoder"]:
            self.register_buffer(
                "title_tokens", torch.from_numpy(inputs.title_tokens), persistent=False
            )
            self.word_embedding = nn.Embedding(
                int(inputs.title_tokens.max()) + 1, MODEL_DIM, padding_idx=0
            )
            self.word_attention = nn.MultiheadAttention(
                MODEL_DIM,
                ATTENTION_HEADS,
                dropout=ATTENTION_DROPOUT,
                batch_first=True,
            )
            self.word_pool = AdditiveAttention(MODEL_DIM)
            width += MODEL_DIM

        self.output = nn.Sequential(
            nn.Linear(width, MODEL_DIM), nn.LayerNorm(MODEL_DIM)
        )

    def encode_titles(self, rows):
        tokens = self.title_tokens[rows]
        padding = tokens.eq(0)
        padding[padding.all(dim=1), 0] = False
        words = self.word_embedding(tokens)
        attended, _ = self.word_attention(
            words, words, words, key_padding_mask=padding, need_weights=False
        )
        return self.word_pool(attended, padding)

    def forward(self, rows):
        unique_rows, inverse = torch.unique(rows, return_inverse=True)
        parts = [self.text[unique_rows]]
        if self.config["metadata"]:
            parts += [
                self.category_embedding(self.category[unique_rows]),
                self.subcategory_embedding(self.subcategory[unique_rows]),
                self.entity_projection(self.entity[unique_rows]),
            ]
        if self.config["title_encoder"]:
            parts.append(self.encode_titles(unique_rows))
        return self.output(torch.cat(parts, dim=-1))[inverse]


class UserEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        if config["position"]:
            self.position_embedding = nn.Embedding(HISTORY_CAP, MODEL_DIM)
        self.self_attention = nn.MultiheadAttention(
            MODEL_DIM,
            ATTENTION_HEADS,
            dropout=ATTENTION_DROPOUT,
            batch_first=True,
        )
        self.norm1 = nn.LayerNorm(MODEL_DIM)
        self.feed_forward = nn.Sequential(
            nn.Linear(MODEL_DIM, 4 * MODEL_DIM),
            nn.ReLU(),
            nn.Dropout(ATTENTION_DROPOUT),
            nn.Linear(4 * MODEL_DIM, MODEL_DIM),
        )
        self.norm2 = nn.LayerNorm(MODEL_DIM)
        self.pool = AdditiveAttention(MODEL_DIM)
        self.default_user = nn.Parameter(torch.zeros(MODEL_DIM))
        if config["short_term"]:
            self.short_term = nn.GRU(MODEL_DIM, MODEL_DIM, batch_first=True)
            self.gate = nn.Linear(2 * MODEL_DIM, MODEL_DIM)
        self.last_gate = None

    @property
    def output_dim(self):
        return MODEL_DIM * (2 if self.config["candidate_pooling"] else 1)

    def forward(self, history, padding_mask, candidates):
        empty = padding_mask.all(dim=1)
        safe_mask = padding_mask.clone()
        safe_mask[empty, -1] = False

        if self.config["position"]:
            ranks = torch.arange(HISTORY_CAP - 1, -1, -1, device=history.device)
            history = history + self.position_embedding(ranks)

        attended, _ = self.self_attention(
            history, history, history, key_padding_mask=safe_mask, need_weights=False
        )
        states = self.norm1(history + attended)
        states = self.norm2(states + self.feed_forward(states))
        states = states.masked_fill(padding_mask.unsqueeze(-1), 0.0)

        user = self.pool(states, padding_mask)
        if self.config["short_term"]:
            recent = states[:, -SHORT_TERM_CAP:]
            _, short = self.short_term(recent)
            short = short.squeeze(0)
            gate = torch.sigmoid(self.gate(torch.cat([user, short], dim=1)))
            user = gate * user + (1 - gate) * short
            self.last_gate = gate.detach().mean().item()
        user = torch.where(empty.unsqueeze(1), self.default_user.unsqueeze(0), user)
        user = user.unsqueeze(1).expand(-1, candidates.shape[1], -1)

        if not self.config["candidate_pooling"]:
            return user
        scores = candidates @ states.transpose(1, 2) / np.sqrt(MODEL_DIM)
        scores = scores.masked_fill(safe_mask.unsqueeze(1), -1e9)
        weights = torch.softmax(scores, dim=-1).masked_fill(
            padding_mask.unsqueeze(1), 0.0
        )
        candidate_user = weights @ states
        return torch.cat([user, candidate_user], dim=-1)


class BranchReranker(nn.Module):
    def __init__(self, config, tabular_width, inputs):
        super().__init__()
        if config["user_encoder"] and not config["text"]:
            raise ValueError("The user encoder needs the news encoder (text=True)")
        if not (config["tabular"] or config["text"]):
            raise ValueError("At least one of the tabular or text branches must be on")
        self.config = config
        width = tabular_width if config["tabular"] else 0
        if config["text"]:
            self.news_encoder = NewsEncoder(config, inputs)
            width += MODEL_DIM
        if config["user_encoder"]:
            self.user_encoder = UserEncoder(config)
            width += self.user_encoder.output_dim

        layers = []
        for hidden in HIDDEN_DIMS:
            layers.extend([nn.Linear(width, hidden), nn.ReLU(), nn.Dropout(DROPOUT)])
            width = hidden
        layers.append(nn.Linear(width, 1))
        self.scorer = nn.Sequential(*layers)

    def forward(self, tabular, candidate_rows, history_rows):
        parts = [tabular] if self.config["tabular"] else []
        if self.config["text"]:
            candidates = self.news_encoder(candidate_rows)
            parts.append(candidates)
            if self.config["user_encoder"]:
                history = self.news_encoder(history_rows)
                parts.append(self.user_encoder(history, history_rows.eq(0), candidates))
        return self.scorer(torch.cat(parts, dim=-1)).squeeze(-1)


def load_profiles(split, impression_ids=None):
    profiles = pl.scan_parquet(RUN / f"request_profiles_{split}" / "*.parquet").select(
        "impression_uid", "full_history_rows"
    )
    if impression_ids is not None:
        profiles = profiles.filter(pl.col("impression_uid").is_in(list(impression_ids)))
    profiles = profiles.collect()
    return dict(
        zip(
            profiles["impression_uid"].to_list(),
            profiles["full_history_rows"].to_list(),
        )
    )


def encode_histories(impression_ids, profiles):
    histories = np.zeros((len(impression_ids), HISTORY_CAP), dtype=np.int64)
    for index, uid in enumerate(impression_ids):
        history = profiles[uid][-HISTORY_CAP:]
        if len(history):
            histories[index, -len(history) :] = history
    return histories


def replay_candidate_features(split, candidates_frame):
    impression_count = candidates_frame["impression_uid"].n_unique()
    cache_name = (
        f"{split}_top{CANDIDATE_CAP}_{impression_count}imp_"
        f"{candidates_frame.height}rows_replay.parquet"
    )
    cached = REPLAY_CACHE_DIR / cache_name
    if RESUME and cached.exists():
        print("Reusing replayed features:", cached)
        return cached

    frame = (
        candidates_frame.sort(["impression_uid", "cutoff"])
        .to_pandas()
        .reset_index(drop=True)
    )
    rows_by_impression = frame.groupby("impression_uid", sort=False).indices
    target_ids = set(rows_by_impression)
    replay_splits = tuple(
        ("train", "validation", "test")[
            : ("train", "validation", "test").index(split) + 1
        ]
    )

    bundle = Bundle(REPLAY_ROOT, DEFAULT_SETTINGS)
    state = TemporalState(bundle, *time_horizon(REPLAY_ROOT), DEFAULT_SETTINGS)
    cache_path = REPLAY_CACHE_DIR / cache_name
    partial_path = cache_path.with_suffix(".partial.parquet")
    writer, pending, impressions, started = None, [], 0, time.monotonic()

    def flush():
        nonlocal writer, pending
        if not pending:
            return
        out = pd.concat(pending, ignore_index=True)
        out = out[["impression_uid", "news_code", "cutoff", *FEATURES]]
        table = pa.Table.from_pandas(out, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(partial_path, table.schema, compression="zstd")
        writer.write_table(table)
        pending = []

    print(
        f"Replaying {split}: {len(frame):,} candidates for "
        f"{len(rows_by_impression):,} impressions"
    )
    try:
        for timestamp, requests, events in replay_batches(
            REPLAY_ROOT, DEFAULT_SETTINGS, splits=replay_splits
        ):
            state.begin(requests, timestamp)
            selected = requests[
                requests["split"].eq(split)
                & requests["impression_uid"].isin(target_ids)
            ]
            if len(selected):
                pairs = frame.iloc[
                    np.concatenate(
                        [rows_by_impression[uid] for uid in selected["impression_uid"]]
                    )
                ]
                features = build_features(
                    selected, pairs, state, bundle, DEFAULT_SETTINGS, "retrieved"
                )
                pending.append(
                    features.merge(
                        pairs,
                        on=["impression_uid", "news_code"],
                        validate="one_to_one",
                    )
                )
                impressions += len(selected)
                if sum(len(chunk) for chunk in pending) >= 50_000:
                    flush()
                    print(
                        f"  {impressions:,}/{len(rows_by_impression):,} impressions "
                        f"({(time.monotonic() - started) / 60:.1f} min)",
                        flush=True,
                    )
            state.finish(requests, events, timestamp, set(replay_splits))
        flush()
    finally:
        if writer is not None:
            writer.close()
        state.close()

    if impressions != len(rows_by_impression):
        raise ValueError(
            f"Replayed {impressions:,} impressions but "
            f"{len(rows_by_impression):,} were selected"
        )
    partial_path.replace(cache_path)
    print("Saved replayed features:", cache_path)
    return cache_path


inputs = load_article_inputs()

if not RR4_LOADED:
    train_slates_pl = pl.from_pandas(train_slates).with_columns(
        pl.col("cutoff").cast(pl.Int16)
    )
    train_features_path = replay_candidate_features(TRAIN_SPLIT, train_slates_pl)
    train_replayed = (
        pl.read_parquet(
            train_features_path,
            columns=["impression_uid", "news_code", "cutoff", *FEATURES],
        )
        .with_columns(pl.col("cutoff").cast(pl.Int16))
        .join(
            train_slates_pl.select("impression_uid", "news_code", "clicked"),
            on=["impression_uid", "news_code"],
            how="inner",
        )
        .sort(["impression_uid", "cutoff"])
    )
    if train_replayed.height != len(train_slates):
        raise ValueError(
            "Replayed features do not cover every generated training candidate"
        )
    train_replayed = train_replayed.to_pandas()

    impression_ids = train_replayed["impression_uid"].drop_duplicates().to_numpy()
    holdout_ids = np.random.default_rng(SEED).choice(
        impression_ids,
        size=max(1, int(HOLDOUT_FRACTION * len(impression_ids))),
        replace=False,
    )
    holdout = train_replayed[
        train_replayed["impression_uid"].isin(holdout_ids)
    ].reset_index(drop=True)
    fit = train_replayed[
        ~train_replayed["impression_uid"].isin(holdout_ids)
    ].reset_index(drop=True)
    print(
        f"Fit: {fit['impression_uid'].nunique():,} impressions / {len(fit):,} candidates; "
        f"holdout: {holdout['impression_uid'].nunique():,} impressions"
    )
else:
    train_slates = None
    train_features_path = None

Reusing replayed features: C:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_03_Pipeline_Evaluation\artifacts\replayed_features\train_top450_4456imp_445600rows_replay.parquet
Fit: 4,011 impressions / 401,100 candidates; holdout: 445 impressions


## 4.3 Preprocessor and Impression Batches

In [30]:
def build_preprocessor():
    numeric = Pipeline(
        [
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]
    )
    categorical = Pipeline(
        [
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )
    return ColumnTransformer(
        [
            ("numeric", numeric, NUMERIC),
            ("categorical", categorical, CATEGORICAL),
        ]
    )


if RR4_LOADED:
    preprocessor = joblib.load(PREPROCESSOR_PATH)
    TABULAR_WIDTH = RR4_CHECKPOINT_META["tabular_width"]
    print("Loaded preprocessor:", PREPROCESSOR_PATH)
else:
    preprocessor = build_preprocessor().fit(fit[FEATURES])
    joblib.dump(preprocessor, PREPROCESSOR_PATH)
    TABULAR_WIDTH = preprocessor.transform(fit[FEATURES].head(1)).shape[1]
    print("Tabular width:", TABULAR_WIDTH)


def build_impression_data(frame, precompute_tabular=True):
    """Arrays for padded impression batches over a training frame."""
    uids = frame["impression_uid"].drop_duplicates().tolist()
    candidate_rows = frame["news_code"].map(inputs.news_to_row)
    if candidate_rows.isna().any():
        raise ValueError("A candidate is missing its article row")
    return {
        "groups": [
            rows.astype(np.int64)
            for rows in frame.groupby("impression_uid", sort=False).indices.values()
        ],
        "histories": torch.from_numpy(
            encode_histories(uids, load_profiles(TRAIN_SPLIT, uids))
        ),
        "tabular": (
            torch.from_numpy(
                preprocessor.transform(frame[FEATURES]).astype(np.float32, copy=False)
            )
            if precompute_tabular
            else None
        ),
        "features": None
        if precompute_tabular
        else frame[FEATURES].reset_index(drop=True),
        "candidates": torch.from_numpy(candidate_rows.to_numpy(np.int64)),
        "labels": torch.from_numpy(frame["clicked"].to_numpy(np.float32)),
    }


def impression_batches(data, order):
    """Yield (tabular, candidates, histories, labels, mask) for batches of impressions."""
    for start in range(0, len(order), BATCH_IMPRESSIONS):
        batch = order[start : start + BATCH_IMPRESSIONS]
        groups = [data["groups"][index] for index in batch]
        width = max(len(group) for group in groups)
        rows = torch.zeros((len(batch), width), dtype=torch.long)
        mask = torch.zeros((len(batch), width), dtype=torch.bool)
        for position, group in enumerate(groups):
            rows[position, : len(group)] = torch.from_numpy(group)
            mask[position, : len(group)] = True
        if data["tabular"] is not None:
            tabular = data["tabular"][rows]
        else:
            raw = data["features"].iloc[rows.reshape(-1).numpy()]
            tabular = torch.from_numpy(
                preprocessor.transform(raw).astype(np.float32, copy=False)
            ).reshape(*rows.shape, -1)
        yield (
            tabular.to(DEVICE),
            data["candidates"][rows].to(DEVICE),
            data["histories"][torch.as_tensor(batch)].to(DEVICE),
            (data["labels"][rows] * mask).to(DEVICE),
            mask.to(DEVICE),
        )


if not RR4_LOADED:
    fit_data = build_impression_data(fit, precompute_tabular=False)
    holdout_data = build_impression_data(holdout, precompute_tabular=False)

Tabular width: 268


## 4.4 Losses and Training Loop

In [31]:
def listwise_softmax_loss(logits, labels, mask):
    """Mean over clicks of -log softmax(click vs. all unclicked candidates in its impression)."""
    positive = (labels > 0) & mask
    negative = (labels == 0) & mask
    negative_lse = torch.logsumexp(
        logits.masked_fill(~negative, float("-inf")), dim=1, keepdim=True
    )
    log_probability = logits - torch.logaddexp(logits, negative_lse)
    return -(log_probability * positive).sum() / positive.sum().clamp_min(1)


def lambda_ndcg_loss(logits, labels, mask, k=NDCG_K):
    """RankNet loss weighted by detached |Delta nDCG@k| for every (click, non-click) pair."""
    masked_logits = logits.masked_fill(~mask, float("-inf"))
    ranks = masked_logits.argsort(dim=1, descending=True).argsort(dim=1).add(1).detach()
    gains = (torch.pow(2.0, labels) - 1.0) * mask
    discounts = (1.0 / torch.log2(ranks.float() + 1.0)) * (ranks <= k) * mask

    ideal_labels = (
        labels.masked_fill(~mask, float("-inf")).sort(dim=1, descending=True).values
    )
    ideal_mask = torch.isfinite(ideal_labels)
    ideal_labels = ideal_labels.clamp_min(0)
    positions = torch.arange(1, logits.shape[1] + 1, device=logits.device)
    ideal_discounts = (1.0 / torch.log2(positions.float() + 1.0)) * (positions <= k)
    idcg = (
        ((torch.pow(2.0, ideal_labels) - 1.0) * ideal_discounts * ideal_mask)
        .sum(dim=1)
        .clamp_min(1e-12)
    )

    score_diff = logits.unsqueeze(2) - logits.unsqueeze(1)
    delta_ndcg = (
        (gains.unsqueeze(2) - gains.unsqueeze(1))
        * (discounts.unsqueeze(2) - discounts.unsqueeze(1))
    ).abs()
    weights = delta_ndcg / idcg[:, None, None]
    valid_pairs = (
        (labels.unsqueeze(2) > labels.unsqueeze(1))
        & mask.unsqueeze(2)
        & mask.unsqueeze(1)
    )
    weights = weights * valid_pairs
    return (F.softplus(-score_diff) * weights).sum() / weights.sum().clamp_min(1e-12)


if RR4_LOADED:
    CLICK_RATE = 0.01
else:
    CLICK_RATE = fit["clicked"].mean()
POSITIVE_WEIGHT = torch.tensor(
    float((1 - CLICK_RATE) / CLICK_RATE), dtype=torch.float32, device=DEVICE
)


def weighted_bce_loss(logits, labels, mask):
    """Pointwise BCE over real candidates with a training-derived positive weight."""
    loss = F.binary_cross_entropy_with_logits(
        logits, labels, pos_weight=POSITIVE_WEIGHT, reduction="none"
    )
    return (loss * mask).sum() / mask.sum()


def bpr_loss(logits, labels, mask):
    """Mean -log sigmoid(s+ - s-) over every (click, non-click) pair within each impression."""
    positive = (labels > 0) & mask
    negative = (labels == 0) & mask
    pairs = positive.unsqueeze(2) & negative.unsqueeze(1)
    score_diff = logits.unsqueeze(2) - logits.unsqueeze(1)
    return (F.softplus(-score_diff) * pairs).sum() / pairs.sum().clamp_min(1)


BASE_LOSSES = {
    "softmax": listwise_softmax_loss,
    "lambda_ndcg": lambda_ndcg_loss,
    "bce": weighted_bce_loss,
    "bpr": bpr_loss,
}


def combined_loss(**weights):
    """Weighted sum of base losses, e.g. combined_loss(softmax=0.8, bce=0.2)."""

    def loss(logits, labels, mask):
        return sum(
            weight * BASE_LOSSES[name](logits, labels, mask)
            for name, weight in weights.items()
        )

    return loss


LOSSES = {
    **BASE_LOSSES,
    "softmax+bce": combined_loss(softmax=0.8, bce=0.2),
    "softmax+lambda_ndcg": combined_loss(softmax=0.5, lambda_ndcg=0.5),
    "lambda_ndcg+bce": combined_loss(lambda_ndcg=0.8, bce=0.2),
    "bpr+bce": combined_loss(bpr=0.75, bce=0.25),
}


def batch_ndcg(logits, labels, mask, k=NDCG_K):
    """Mean nDCG@k over a batch of impressions (retained candidates only)."""
    order = logits.masked_fill(~mask, float("-inf")).argsort(dim=1, descending=True)[
        :, :k
    ]
    discounts = 1.0 / torch.log2(
        torch.arange(2, order.shape[1] + 2, device=logits.device).float()
    )
    dcg = (labels.gather(1, order) * discounts).sum(dim=1)
    ideal = (
        labels.masked_fill(~mask, 0)
        .sort(dim=1, descending=True)
        .values[:, : order.shape[1]]
        * discounts
    ).sum(dim=1)
    return (dcg / ideal.clamp_min(1e-12)).mean().item()


def state_is_finite(model):
    return all(
        bool(torch.isfinite(value).all())
        for value in model.state_dict().values()
        if value.is_floating_point()
    )


def train_model(
    config, loss_name, fit_data, holdout_data, epochs=EPOCHS, patience=PATIENCE
):
    """Train one architecture with one loss; return the best-holdout-epoch model."""
    torch.manual_seed(SEED)
    model = BranchReranker(config, TABULAR_WIDTH, inputs).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    loss_fn = LOSSES[loss_name]
    rng = np.random.default_rng(SEED)
    best = {"ndcg": -1.0, "epoch": 0, "state": None}

    for epoch in range(1, epochs + 1):
        model.train()
        losses = []
        skipped = 0
        for tabular, candidates_batch, histories, labels, mask in impression_batches(
            fit_data, rng.permutation(len(fit_data["groups"]))
        ):
            optimizer.zero_grad()
            loss = loss_fn(model(tabular, candidates_batch, histories), labels, mask)
            if not torch.isfinite(loss):
                skipped += 1
                optimizer.zero_grad()
                continue
            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            if not torch.isfinite(grad_norm):
                skipped += 1
                optimizer.zero_grad()
                continue
            optimizer.step()
            losses.append(loss.item())

        if not losses:
            print(f"  epoch {epoch}/{epochs}: no finite batches; stopping")
            break

        model.eval()
        holdout_ok = state_is_finite(model)
        with torch.no_grad():
            holdout_scores = []
            if holdout_ok:
                for (
                    tabular,
                    candidates_batch,
                    histories,
                    labels,
                    mask,
                ) in impression_batches(
                    holdout_data, np.arange(len(holdout_data["groups"]))
                ):
                    logits = model(tabular, candidates_batch, histories)
                    if not torch.isfinite(logits).all():
                        holdout_ok = False
                        break
                    holdout_scores.append(batch_ndcg(logits, labels, mask))
        holdout_ndcg = (
            float(np.mean(holdout_scores))
            if holdout_ok and holdout_scores
            else float("-inf")
        )
        print(
            f"  epoch {epoch}/{epochs}: {loss_name} loss {np.mean(losses):.4f}, "
            f"holdout nDCG@{NDCG_K} {holdout_ndcg:.4f}"
        )
        if holdout_ndcg > best["ndcg"]:
            best = {
                "ndcg": holdout_ndcg,
                "epoch": epoch,
                "state": {
                    key: value.detach().cpu().clone()
                    for key, value in model.state_dict().items()
                },
            }
        elif patience is not None and epoch - best["epoch"] >= patience:
            print(
                f"  early stopping: no holdout improvement for {patience} epochs; "
                f"keeping epoch {best['epoch']}"
            )
            break

    if best["state"] is None:
        raise ValueError("Training produced no finite model with a valid holdout score")
    model.load_state_dict(best["state"])
    model.best_epoch, model.best_holdout_ndcg = best["epoch"], best["ndcg"]
    return model

## 4.5 Train or Load RR4

In [32]:
if RR4_LOADED:
    model = BranchReranker(CONFIG, RR4_CHECKPOINT_META["tabular_width"], inputs).to(
        DEVICE
    )
    model.load_state_dict(RR4_CHECKPOINT_META["state_dict"])
    model.eval()
    model.best_epoch = RR4_CHECKPOINT_META.get("best_epoch")
    model.best_holdout_ndcg = RR4_CHECKPOINT_META.get("holdout_ndcg")
    print("Loaded RR4:", RR4_CHECKPOINT_PATH)
else:
    torch.manual_seed(SEED)
    model = train_model(
        CONFIG, LOSS_NAME, fit_data, holdout_data, epochs=EPOCHS, patience=PATIENCE
    )
    if not state_is_finite(model):
        raise ValueError("Refusing to save a reranker with non-finite weights")
    torch.save(
        {
            "state_dict": {
                key: value.detach().cpu() for key, value in model.state_dict().items()
            },
            "config": CONFIG,
            "loss": LOSS_NAME,
            "tabular_width": TABULAR_WIDTH,
            "features": FEATURES,
            "train_impressions": TRAIN_IMPRESSIONS,
            "best_epoch": int(model.best_epoch),
            "holdout_ndcg": float(model.best_holdout_ndcg),
        },
        RR4_CHECKPOINT_PATH,
    )
    print("Saved RR4:", RR4_CHECKPOINT_PATH)

  epoch 1/3: softmax+lambda_ndcg loss 1.9049, holdout nDCG@10 0.4590
  epoch 2/3: softmax+lambda_ndcg loss 1.5861, holdout nDCG@10 0.4858
  epoch 3/3: softmax+lambda_ndcg loss 1.5238, holdout nDCG@10 0.4750
Saved RR4: C:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_03_Pipeline_Evaluation\artifacts\attention_reranker_niche\reranker\model.pt


# 5. Score the Target Candidates

In [33]:
def score_candidates(model, split, candidate_frame):
    features_path = replay_candidate_features(split, candidate_frame)
    replayed = (
        pl.read_parquet(
            features_path,
            columns=["impression_uid", "news_code", "cutoff", *FEATURES],
        )
        .with_columns(pl.col("cutoff").cast(pl.Int16))
        .join(
            candidate_frame.select(
                "impression_uid", "news_code", "cutoff"
            ).with_columns(pl.col("cutoff").cast(pl.Int16)),
            on=["impression_uid", "news_code", "cutoff"],
            how="inner",
        )
        .sort(["impression_uid", "cutoff"])
    )
    if replayed.height != candidate_frame.height:
        raise ValueError(
            "The replay cache does not cover every selected candidate; "
            "delete the matching cache file and rerun."
        )

    profiles = load_profiles(
        split, candidate_frame["impression_uid"].unique().to_list()
    )
    uids = replayed["impression_uid"].unique().to_list()
    histories = encode_histories(uids, profiles)
    uid_index = {uid: index for index, uid in enumerate(uids)}
    history_index = np.fromiter(
        (uid_index[uid] for uid in replayed["impression_uid"].to_list()),
        dtype=np.int64,
        count=replayed.height,
    )
    feature_frame = replayed.select(FEATURES).to_pandas()
    news_codes = replayed["news_code"].to_list()
    missing_codes = [code for code in news_codes if code not in inputs.news_to_row]
    if missing_codes:
        raise ValueError(
            f"{len(missing_codes)} candidates are missing article rows, e.g. {missing_codes[:5]}"
        )
    candidate_rows = np.fromiter(
        (inputs.news_to_row[code] for code in news_codes),
        dtype=np.int64,
        count=len(news_codes),
    )

    model.eval()
    scores = []
    with torch.no_grad():
        for start in range(0, replayed.height, SCORING_BATCH_ROWS):
            stop = min(start + SCORING_BATCH_ROWS, replayed.height)
            tabular = preprocessor.transform(feature_frame.iloc[start:stop]).astype(
                np.float32, copy=False
            )
            logits = model(
                torch.from_numpy(tabular).unsqueeze(1).to(DEVICE),
                torch.from_numpy(candidate_rows[start:stop]).unsqueeze(1).to(DEVICE),
                torch.from_numpy(histories[history_index[start:stop]]).to(DEVICE),
            )
            batch_scores = torch.sigmoid(logits.squeeze(1))
            if not torch.isfinite(batch_scores).all():
                raise ValueError(
                    "The reranker produced non-finite scores; retrain the model"
                )
            scores.append(batch_scores.cpu().numpy())
    return replayed.select("impression_uid", "news_code", "cutoff").with_columns(
        pl.Series("score", np.concatenate(scores), dtype=pl.Float32)
    )


scores = score_candidates(model, TARGET_SPLIT, target_candidate_frame)
if not scores["score"].is_finite().all():
    raise ValueError("The reranker produced non-finite scores")
scored_candidates = target_candidate_frame.join(
    scores.select("impression_uid", "news_code", "score"),
    on=["impression_uid", "news_code"],
    how="left",
)
if scored_candidates["score"].null_count():
    raise ValueError("Some selected candidates have no reranker score")
print(
    f"Scored candidates: {scored_candidates.height:,} rows, "
    f"{scored_candidates['impression_uid'].n_unique():,} impressions"
)

Reusing replayed features: C:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_03_Pipeline_Evaluation\artifacts\replayed_features\test_top450_30270imp_13621500rows_replay.parquet
Scored candidates: 13,621,500 rows, 30,270 impressions


# 6. Form the Final Top-K Lists and Evaluate

In [34]:
if not scored_candidates["score"].is_finite().all():
    raise ValueError("Cannot rank candidates with non-finite reranker scores")
reranker_top_k = (
    scored_candidates.sort(
        ["impression_uid", "score", "news_code"], descending=[False, True, False]
    )
    .with_columns(
        pl.int_range(1, pl.len() + 1)
        .over("impression_uid")
        .cast(pl.Int16)
        .alias("cutoff")
    )
    .filter(pl.col("cutoff") <= K)
    .select("impression_uid", "cutoff", "news_code", "score")
)

top_shape = reranker_top_k.group_by("impression_uid").agg(
    pl.len().alias("rows"),
    pl.col("cutoff").min().alias("first"),
    pl.col("cutoff").max().alias("last"),
    pl.col("cutoff").n_unique().alias("ranks"),
    pl.col("news_code").n_unique().alias("articles"),
)
bad = top_shape.filter(
    (pl.col("rows") != K)
    | (pl.col("first") != 1)
    | (pl.col("last") != K)
    | (pl.col("ranks") != K)
    | (pl.col("articles") != K)
)
if bad.height:
    raise ValueError(
        f"attention_reranker has an invalid Top-K shape for {bad.height} impressions"
    )
print("Reranker Top-K rows:", reranker_top_k.height)

Reranker Top-K rows: 908100


In [35]:
cg_evaluation.ONE_HOUR = pl.lit(timedelta(hours=1)).cast(pl.Duration("ns"))

evaluator = CandidateEvaluator(root=REPLAY_ROOT, split=TARGET_SPLIT)
evaluator._hourly = evaluator.hourly.with_columns(
    pl.col("hour").cast(pl.Datetime("ns"))
)

CG_METRICS = [
    "recall_pooled",
    "recall_impression_mean",
    "hit_rate",
    "long_tail_recall",
    "cold_article_recall",
    "novelty_bits",
    "catalogue_coverage",
    "personalisation",
]


def evaluate_pipeline(top_k, model_name, k):
    # Score only the first k articles of the ranked list
    top_k = top_k.filter(pl.col("cutoff") <= k)
    cg = evaluator.evaluate(
        top_k.select("impression_uid", "cutoff", "news_code"),
        model_name=model_name,
        m_values=(k,),
        impression_uids=selected_uids,
    )
    cg_wide = (
        cg.pivot(on="metric", index=["model", "split", "m"], values="value")
        .rename({"m": "k"})
        .with_columns(pl.lit(n_selected).alias("impressions"))
    )
    rr = evaluate_retrieved_candidates(
        top_k.select("impression_uid", "news_code", "score"),
        feature_run=RUN,
        split=TARGET_SPLIT,
        score_column="score",
        model_name=model_name,
        k_values=(k,),
    ).select(
        "model",
        pl.col("impressions").alias("clicked_impressions"),
        "mrr",
        pl.col(f"ndcg_at_{k}").alias("ndcg"),
    )
    return cg_wide.join(rr, on="model", how="left")


# Score the reranked list at every cutoff in K_VALUES
results = pl.concat(
    [evaluate_pipeline(reranker_top_k, "attention_reranker", k) for k in K_VALUES],
    how="diagonal_relaxed",
)
ordered = results.select(
    "model",
    "split",
    "k",
    "impressions",
    "clicked_impressions",
    "ndcg",
    "mrr",
    *CG_METRICS,
).sort("k")
display(ordered)

model,split,k,impressions,clicked_impressions,ndcg,mrr,recall_pooled,recall_impression_mean,hit_rate,long_tail_recall,cold_article_recall,novelty_bits,catalogue_coverage,personalisation
str,str,i64,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""attention_reranker""","""test""",5,30270,30270,0.120153,0.121152,0.146332,0.167478,0.215593,0.008538,0.0,11.337718,0.008853,0.839821
"""attention_reranker""","""test""",10,30270,30270,0.148234,0.134273,0.222723,0.248887,0.31513,0.011213,0.0,11.523846,0.013845,0.827499
"""attention_reranker""","""test""",30,30270,30270,0.197798,0.146486,0.409909,0.441484,0.53069,0.019854,0.0,11.925909,0.025662,0.783095


# 7. Save Outputs

In [36]:
final_top_k_path = (
    OUTPUT_DIR / f"{BASE_MODEL_NAME}_{TARGET_SPLIT}_top{K}_{IMPRESSION_TAG}.parquet"
)
reranker_top_k.write_parquet(final_top_k_path)
print("Final Top-K saved to:", final_top_k_path)

results_path = (
    RESULTS / f"pipeline_{BASE_MODEL_NAME}_{TARGET_SPLIT}_{IMPRESSION_TAG}.csv"
)
ordered.write_csv(results_path)
print("Metrics saved to:", results_path)

metadata_path = (
    RESULTS / f"pipeline_{BASE_MODEL_NAME}_{TARGET_SPLIT}_{IMPRESSION_TAG}.json"
)
metadata_path.write_text(
    json.dumps(
        {
            "pipeline": "popular_niche_combined (niche two-tower) + generated-candidate attention reranker",
            "split": TARGET_SPLIT,
            "k": list(K_VALUES),
            "eval_impressions_requested": EVAL_IMPRESSIONS,
            "eval_impressions": int(n_selected),
            "target_candidate_rows": int(target_candidate_frame.height),
            "seed": SEED,
            "cg_checkpoint": str(CG_CHECKPOINT_PATH),
            "cg_reused": bool(CG_LOADED),
            "cg_best_epoch": CG_BEST_EPOCH,
            "rr4_checkpoint": str(RR4_CHECKPOINT_PATH),
            "rr4_preprocessor": str(PREPROCESSOR_PATH),
            "rr4_reused": bool(RR4_LOADED),
            "rr4_best_epoch": int(model.best_epoch)
            if model.best_epoch is not None
            else None,
            "train_impressions_requested": TRAIN_IMPRESSIONS,
            "train_slates": int(train_slates["impression_uid"].nunique())
            if train_slates is not None
            else None,
            "train_rows": len(train_slates) if train_slates is not None else None,
            "max_candidates": MAX_CANDIDATES,
            "architecture": CONFIG,
            "loss": LOSS_NAME,
            "metrics": {
                f"{row['model']}@{row['k']}": {"ndcg": row["ndcg"], "mrr": row["mrr"]}
                for row in ordered.iter_rows(named=True)
            },
        },
        indent=2,
    )
)
print("Metadata saved to:", metadata_path)

Final Top-K saved to: C:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_03_Pipeline_Evaluation\artifacts\attention_reranker_niche_generated_test_top30_all.parquet
Metrics saved to: C:\Users\Yong Rui\BT4222-Group-6-\results\pipeline_attention_reranker_niche_generated_test_all.csv
Metadata saved to: C:\Users\Yong Rui\BT4222-Group-6-\results\pipeline_attention_reranker_niche_generated_test_all.json



# 7. Notes and Limitations

- The candidate generator is trained inside this notebook on logged clicks, then used to build
  the training and target candidate lists in memory. No candidate parquet or score parquet is
  written.
- The candidate-generator checkpoint and the RR4 checkpoint are reused when present and
  compatible with the current config/schema. Delete the relevant `model.pt` to force retraining.
- Training uses all generated candidates for the sampled training impressions. Candidates that
  were never shown are treated as zero-labelled negatives, which can introduce false negatives.
- Validation and test treat unshown candidates as unknown: a clicked article the generator
  missed stays in the denominator and counts as a miss.
- `TRAIN_IMPRESSIONS` and `EVAL_IMPRESSIONS` control runtime. `None` uses every impression.
  The causal-feature replay cache is reused across runs when the sampled candidate file matches.
- `cg_evaluation.py` metrics at `M = K` describe the final recommendation list, not the
  450-candidate pool. `catalogue_coverage` and `personalisation` therefore measure the final
  Top-K, not retrieval depth.
- For a test run, set `TARGET_SPLIT = "test"` and rerun. The generator is reused from its
  checkpoint when available; the reranker is reused from its checkpoint when available.